# Rank-1 LoRA on `down_proj` at block 23

Trains one rank-1 adapter on the 56 **desperate** training stories, and a second one on the 56 **calm** stories as a control. Then it checks the three things the prof's note asks about:

1. **Filter:** does A·x fire more on desperate stories than on other emotions? (validation split, 12 stories per emotion)
2. **Steer:** does B point along the desperation vector more than along the other 13?
3. **Control:** what happens to the model's writing when we scale the adapter ×0, ×1, ×4?

**Run on:** an NVIDIA GPU (tested on a Brev A4000, 16 GB), from inside `notebooks/`. Needs the same files as `steering.ipynb`.

In [1]:
import gc, json, sys
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F

#let the notebook import our code from src/
sys.path.insert(0, str(Path.cwd().parent))
from src.steering import LAYER, load_model_8b, generate, residual_norms
from src.lora import add_adapter, adapter_weights, down_proj, firing, text_loss, train_adapter

RUN_DIR = Path("../emotion_vector_pipeline/outputs/emotion_extraction/ab1333f49024705c")
STORIES = Path("../emotion_vector_pipeline/data/emotion_stories_qwen_full1120.jsonl")
OUT = Path("../results/lora")
OUT.mkdir(parents=True, exist_ok=True)
ADAPTERS = Path("outputs/lora_adapters")  #gitignored; the adapters themselves are small but binary
PROMPT = "Write a short paragraph about your morning."  #same prompt as steering.ipynb

rows = [json.loads(line) for line in STORIES.open()]
def stories(emotion, split):
    return [r["text"] for r in rows if r["emotion"] == emotion and r["split"] == split]

with np.load(RUN_DIR / "emotion_vectors.npz", allow_pickle=False) as saved:
    EMOTIONS = saved["emotion_names"].tolist()
    VEC = torch.tensor(saved["unit_vectors"][:, saved["layer_indices"].tolist().index(LAYER)])
print(len(rows), VEC.shape)  #expect 1120 torch.Size([14, 4096])
print({e: len(stories(e, "train")) for e in ["desperate", "calm"]})  #expect 56 each

#how big "lined up" has to be: cosine of random directions with the desperation vector
rand = torch.randn(2000, VEC.shape[1], generator=torch.Generator().manual_seed(0))
rand_cos = F.cosine_similarity(rand, VEC[EMOTIONS.index("desperate")][None], dim=1)
print("random directions: cosine std", round(rand_cos.std().item(), 4))  #expect about 1/64 = 0.016

1120 torch.Size([14, 4096])
{'desperate': 56, 'calm': 56}
random directions: cosine std 0.0154


## Train one adapter and read it out

In [2]:
def run_one(emotion, epochs=4, lr=1e-3):
    tok, base = load_model_8b()
    model = add_adapter(base)
    model.print_trainable_parameters()  #expect 16,384 trainable (12,288 in A + 4,096 in B)

    train, val = stories(emotion, "train"), stories(emotion, "validation")
    with model.disable_adapter():
        before = text_loss(model, tok, val)
        baseline_text = generate(model, tok, PROMPT, seed=0)
    losses = train_adapter(model, tok, train, epochs=epochs, lr=lr)
    after = text_loss(model, tok, val)
    print(f"{emotion} validation loss: {before:.3f} -> {after:.3f}")

    A, B, scaling = adapter_weights(model)

    #filter: mean A·x on each emotion's 12 validation stories
    fire = {e: torch.cat(firing(model, tok, stories(e, "validation"))).mean().item() for e in EMOTIONS}

    #A and B can both flip sign without changing B·A, so point B the way it pushes on its own emotion
    sign = 1.0 if fire[emotion] >= 0 else -1.0
    cos = {e: F.cosine_similarity(sign * B, VEC[i], dim=0).item() for i, e in enumerate(EMOTIONS)}

    #how hard it pushes, in the same units as steering strength (fraction of the residual norm)
    #residual_norms hooks the plain Qwen model; the adapter is off so we measure the original size
    with model.disable_adapter():
        _, rest = residual_norms(base, tok, [stories(e, "validation")[0] for e in EMOTIONS])
    strength = abs(fire[emotion]) * scaling * B.norm().item() / rest.mean().item()

    #control: scale the adapter and write with the same prompt and seed
    mod, gens = down_proj(model), {}
    for s in [0, 1, 4]:
        mod.scaling["default"] = scaling * s
        gens[str(s)] = generate(model, tok, PROMPT, seed=0)
    mod.scaling["default"] = scaling
    assert gens["0"] == baseline_text, "scale 0 should match the model with the adapter off"

    model.save_pretrained(ADAPTERS / emotion)
    result = dict(
        emotion=emotion, epochs=epochs, lr=lr, steps=len(losses),
        val_loss_before=before, val_loss_after=after, train_losses=losses,
        firing_by_emotion=fire, cosine_B_vs_vectors=cos, equivalent_strength=strength,
        B_norm=B.norm().item(), top_A_neurons=A.abs().topk(10).indices.tolist(),
        generations=gens,
    )
    del model, base
    gc.collect()
    torch.cuda.empty_cache()
    return result

## Run both adapters

In [3]:
results = {}
for emotion in ["desperate", "calm"]:
    results[emotion] = run_one(emotion)
    (OUT / f"{emotion}_adapter.json").write_text(json.dumps(results[emotion], indent=2))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

trainable params: 16,384 || all params: 8,190,751,744 || trainable%: 0.0002


epoch 1: mean loss 1.468


epoch 2: mean loss 1.285


epoch 3: mean loss 1.204


... (5 outputs left out)

epoch 2: mean loss 1.351


epoch 3: mean loss 1.267


epoch 4: mean loss 1.214


calm validation loss: 1.619 -> 1.389


## What did they learn?

In [4]:
for emotion, r in results.items():
    print(f"=== {emotion} adapter ===")
    print(f"validation loss {r['val_loss_before']:.3f} -> {r['val_loss_after']:.3f}")
    print("filter, mean A·x by emotion (top 5):")
    for e, z in sorted(r["firing_by_emotion"].items(), key=lambda kv: -abs(kv[1]))[:5]:
        print(f"  {e:11s} {z:+.3f}")
    print("steer, cosine of B with each emotion vector (top 5):")
    for e, c in sorted(r["cosine_B_vs_vectors"].items(), key=lambda kv: -kv[1])[:5]:
        print(f"  {e:11s} {c:+.3f}")
    print(f"pushes like steering strength {r['equivalent_strength']:.3f} on its own emotion")
    for s, text in r["generations"].items():
        print(f"--- scale x{s} ---\n{text}\n")

=== desperate adapter ===
validation loss 1.467 -> 1.276
filter, mean A·x by emotion (top 5):
  afraid      -40.969
  guilty      -40.946
  desperate   -40.469
  surprised   -40.225
  sad         -40.045
steer, cosine of B with each emotion vector (top 5):
  sad         +0.054
  desperate   +0.049
  guilty      +0.042
  angry       +0.031
  afraid      +0.025
pushes like steering strength 0.354 on its own emotion
--- scale x0 ---
My morning starts with the soft glow of dawn filtering through the curtains. I brew a cup of coffee, savoring the quiet moment as I listen to the gentle sounds of the city waking up. I take a few deep breaths, stretch, and let the calm of the early hours settle into my mind. It's a peaceful ritual that sets a positive tone for the day ahead.

--- scale x1 ---
Every morning starts with the soft hum of the city waking up outside my window. I brew a cup of coffee, the steam curling into the air as I sit on the edge of the bed, sipping slowly while the world still